# Caching Strategies for AI Systems

This notebook covers practical caching techniques for AI pipelines:
- Response caching with Redis/Upstash
- Embedding caching for RAG
- Cache hit rate tracking
- Measuring cache effectiveness

## 1. Setup and Dependencies

In [ ]:
import hashlib
import json
import time
import asyncio
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Optional

# For production: use redis.asyncio
# pip install redis
# import redis.asyncio as redis

print("Dependencies loaded.")

## 2. In-Memory Cache Implementation

Start simple before adding Redis. Useful for local dev and single-process deployments.

In [ ]:
class InMemoryCache:
    """Simple in-memory cache with TTL support."""

    def __init__(self, default_ttl: int = 3600):
        self.default_ttl = default_ttl
        self._store: dict[str, tuple[str, float]] = {}  # key -> (value, expiry)
        self._hits = 0
        self._misses = 0

    def _make_key(self, prefix: str, content: str) -> str:
        """Create a deterministic cache key."""
        content_hash = hashlib.sha256(content.encode()).hexdigest()[:16]
        return f"{prefix}:{content_hash}"

    def get(self, prefix: str, content: str) -> Optional[str]:
        """Get value from cache. Returns None on miss."""
        key = self._make_key(prefix, content)
        if key in self._store:
            value, expiry = self._store[key]
            if time.time() < expiry:
                self._hits += 1
                return value
            else:
                del self._store[key]  # expired
        self._misses += 1
        return None

    def set(self, prefix: str, content: str, value: str, ttl: Optional[int] = None):
        """Set value in cache."""
        key = self._make_key(prefix, content)
        expiry = time.time() + (ttl or self.default_ttl)
        self._store[key] = (value, expiry)

    @property
    def hit_rate(self) -> float:
        """Calculate cache hit rate."""
        total = self._hits + self._misses
        return self._hits / total if total > 0 else 0.0

    @property
    def stats(self) -> dict:
        return {
            "hits": self._hits,
            "misses": self._misses,
            "hit_rate": f"{self.hit_rate:.1%}",
            "size": len(self._store),
        }

    def clear(self):
        self._store.clear()
        self._hits = 0
        self._misses = 0


cache = InMemoryCache(default_ttl=300)
print("In-memory cache created.")
print(f"Stats: {cache.stats}")

## 3. Response Caching

Cache LLM responses keyed by prompt hash. Avoids redundant API calls for repeated queries.

In [ ]:
@dataclass
class LLMResponse:
    content: str
    model: str
    tokens_used: int
    latency_ms: float
    cached: bool = False


class ResponseCacher:
    """Cache LLM responses to avoid redundant API calls."""

    def __init__(self, cache: InMemoryCache, model: str = "gpt-4o"):
        self.cache = cache
        self.model = model

    def _cache_key(self, prompt: str, system_prompt: str = "") -> str:
        """Create cache key from prompt + system prompt + model."""
        content = json.dumps({
            "model": self.model,
            "system": system_prompt,
            "prompt": prompt
        }, sort_keys=True)
        return content

    async def get_or_generate(
        self,
        prompt: str,
        generate_fn,
        system_prompt: str = "",
        ttl: int = 3600
    ) -> LLMResponse:
        """Get cached response or generate new one."""
        cache_content = self._cache_key(prompt, system_prompt)
        cached = self.cache.get("llm", cache_content)

        if cached:
            data = json.loads(cached)
            return LLMResponse(
                content=data["content"],
                model=data["model"],
                tokens_used=0,  # no tokens consumed
                latency_ms=0,   # instant
                cached=True
            )

        # Generate new response
        start = time.time()
        response = await generate_fn(prompt, system_prompt)
        latency = (time.time() - start) * 1000

        # Cache the response
        self.cache.set(
            "llm", cache_content,
            json.dumps({
                "content": response.content,
                "model": response.model,
            }),
            ttl=ttl
        )

        return LLMResponse(
            content=response.content,
            model=response.model,
            tokens_used=response.tokens_used,
            latency_ms=latency,
            cached=False
        )


print("ResponseCacher class defined.")

In [ ]:
# Simulate LLM calls
call_count = 0

async def mock_llm_generate(prompt: str, system_prompt: str = "") -> LLMResponse:
    """Simulate an LLM API call with latency."""
    global call_count
    call_count += 1
    await asyncio.sleep(0.1)  # simulate API latency
    return LLMResponse(
        content=f"Response to: {prompt[:50]}...",
        model="gpt-4o",
        tokens_used=len(prompt.split()) + 20,
        latency_ms=100
    )

# Test caching
response_cache = ResponseCacher(InMemoryCache(default_ttl=300))

async def test_response_caching():
    global call_count
    call_count = 0

    queries = [
        "What is the capital of France?",
        "What is the capital of France?",  # repeat
        "What is the capital of France?",  # repeat
        "What is the capital of Germany?",
        "What is the capital of France?",  # repeat
    ]

    results = []
    for query in queries:
        result = await response_cache.get_or_generate(
            query, mock_llm_generate
        )
        results.append(result)
        print(f"Query: {query[:30]:<30} | Cached: {result.cached} | Latency: {result.latency_ms:.0f}ms")

    print(f"\nTotal API calls: {call_count}")
    print(f"Cache stats: {response_cache.cache.stats}")

await test_response_caching()

## 4. Embedding Caching

Cache embeddings to avoid recomputing vectors for repeated text chunks in RAG.

In [ ]:
class EmbeddingCacher:
    """Cache embeddings to avoid redundant computation."""

    def __init__(self, cache: InMemoryCache):
        self.cache = cache
        self._compute_count = 0

    async def get_or_compute(
        self,
        text: str,
        compute_fn,
        ttl: int = 86400  # 24h default
    ) -> list[float]:
        """Get cached embedding or compute new one."""
        cached = self.cache.get("emb", text)

        if cached:
            return json.loads(cached)

        self._compute_count += 1
        embedding = await compute_fn(text)

        self.cache.set("emb", text, json.dumps(embedding), ttl=ttl)
        return embedding

    async def batch_get_or_compute(
        self,
        texts: list[str],
        batch_compute_fn,
        ttl: int = 86400
    ) -> list[list[float]]:
        """Process a batch, only computing embeddings for cache misses."""
        results = [None] * len(texts)
        miss_indices = []
        miss_texts = []

        # Check cache for all texts
        for i, text in enumerate(texts):
            cached = self.cache.get("emb", text)
            if cached:
                results[i] = json.loads(cached)
            else:
                miss_indices.append(i)
                miss_texts.append(text)

        # Compute only misses
        if miss_texts:
            self._compute_count += len(miss_texts)
            new_embeddings = await batch_compute_fn(miss_texts)

            for i, embedding in zip(miss_indices, new_embeddings):
                results[i] = embedding
                self.cache.set("emb", texts[i], json.dumps(embedding), ttl=ttl)

        return results

    @property
    def compute_count(self) -> int:
        return self._compute_count


print("EmbeddingCacher class defined.")

In [ ]:
# Simulate embedding computation
async def mock_embed(text: str) -> list[float]:
    """Simulate embedding computation."""
    await asyncio.sleep(0.05)
    # Deterministic pseudo-embedding based on text
    import random
    rng = random.Random(hash(text))
    return [rng.random() for _ in range(384)]

async def mock_batch_embed(texts: list[str]) -> list[list[float]]:
    """Simulate batch embedding computation."""
    return [await mock_embed(t) for t in texts]


# Test embedding caching
emb_cache = EmbeddingCacher(InMemoryCache(default_ttl=3600))

async def test_embedding_caching():
    documents = [
        "The quick brown fox jumps over the lazy dog.",
        "Python is a versatile programming language.",
        "Machine learning models require large datasets.",
        "The quick brown fox jumps over the lazy dog.",  # duplicate
        "Redis is an in-memory data store.",
        "Python is a versatile programming language.",  # duplicate
    ]

    print(f"Embedding {len(documents)} documents...")
    embeddings = await emb_cache.batch_get_or_compute(documents, mock_batch_embed)

    print(f"Compute count: {emb_cache.compute_count}")
    print(f"Cache stats: {emb_cache.cache.stats}")
    print(f"\nSaved {len(documents) - emb_cache.compute_count} redundant computations.")

await test_embedding_caching()

## 5. Cache Hit Rate Tracking

Monitor cache performance to tune TTLs and understand query patterns.

In [ ]:
class CacheMonitor:
    """Track cache performance metrics over time."""

    def __init__(self):
        self.metrics = defaultdict(lambda: {"hits": 0, "misses": 0, "latencies": []})

    def record(self, cache_type: str, hit: bool, latency_ms: float):
        """Record a cache access."""
        if hit:
            self.metrics[cache_type]["hits"] += 1
        else:
            self.metrics[cache_type]["misses"] += 1
        self.metrics[cache_type]["latencies"].append(latency_ms)

    def summary(self) -> dict:
        """Generate performance summary."""
        summary = {}
        for cache_type, data in self.metrics.items():
            total = data["hits"] + data["misses"]
            latencies = data["latencies"]
            summary[cache_type] = {
                "total_accesses": total,
                "hits": data["hits"],
                "misses": data["misses"],
                "hit_rate": f"{data['hits'] / total:.1%}" if total > 0 else "N/A",
                "avg_latency_ms": f"{sum(latencies) / len(latencies):.1f}" if latencies else "N/A",
            }
        return summary


# Simulate cache access patterns
monitor = CacheMonitor()

# Simulate 100 requests with 40% cache hit probability
import random
random.seed(42)

for i in range(100):
    is_hit = random.random() < 0.4
    latency = random.uniform(1, 5) if is_hit else random.uniform(50, 200)
    monitor.record("response_cache", is_hit, latency)

print("Cache Performance Summary:")
for cache_type, metrics in monitor.summary().items():
    print(f"\n{cache_type}:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")

## 6. Measuring Cache Effectiveness

Compare performance with and without caching.

In [ ]:
async def benchmark_with_caching(
    queries: list[str],
    cache: InMemoryCache,
    generate_fn
) -> dict:
    """Benchmark with caching enabled."""
    cacher = ResponseCacher(cache)
    total_latency = 0
    api_calls = 0

    for query in queries:
        start = time.time()
        result = await cacher.get_or_generate(query, generate_fn)
        total_latency += (time.time() - start) * 1000
        if not result.cached:
            api_calls += 1

    return {
        "total_latency_ms": round(total_latency, 1),
        "avg_latency_ms": round(total_latency / len(queries), 1),
        "api_calls": api_calls,
        "cache_hit_rate": cache.hit_rate,
        "queries": len(queries),
    }

async def benchmark_without_caching(
    queries: list[str],
    generate_fn
) -> dict:
    """Benchmark without caching (baseline)."""
    total_latency = 0

    for query in queries:
        start = time.time()
        await generate_fn(query)
        total_latency += (time.time() - start) * 1000

    return {
        "total_latency_ms": round(total_latency, 1),
        "avg_latency_ms": round(total_latency / len(queries), 1),
        "api_calls": len(queries),
        "cache_hit_rate": 0.0,
        "queries": len(queries),
    }

# Run benchmarks
test_queries = [
    "What is machine learning?",
    "Explain neural networks.",
    "What is machine learning?",
    "What is deep learning?",
    "Explain neural networks.",
    "What is machine learning?",
    "How does backpropagation work?",
    "What is deep learning?",
    "What is machine learning?",
    "Explain neural networks.",
]

print("Running benchmarks...\n")

without_cache = await benchmark_without_caching(test_queries, mock_llm_generate)
with_cache = await benchmark_with_caching(
    test_queries, InMemoryCache(default_ttl=300), mock_llm_generate
)

print("=== Results ===")
print(f"Without cache:")
print(f"  Total latency: {without_cache['total_latency_ms']}ms")
print(f"  API calls: {without_cache['api_calls']}")
print(f"\nWith cache:")
print(f"  Total latency: {with_cache['total_latency_ms']}ms")
print(f"  API calls: {with_cache['api_calls']}")
print(f"  Cache hit rate: {with_cache['cache_hit_rate']:.1%}")

speedup = without_cache['total_latency_ms'] / with_cache['total_latency_ms']
print(f"\nSpeedup: {speedup:.1f}x")
print(f"API calls saved: {without_cache['api_calls'] - with_cache['api_calls']}")

## 7. Redis/Upstash Integration (Production)

For production, use Redis or Upstash for distributed caching across multiple instances.

In [ ]:
# Production Redis integration (requires redis package)
# Uncomment and configure for actual Redis usage

"""
import redis.asyncio as redis

class RedisResponseCache:
    def __init__(self, redis_url: str = "redis://localhost:6379", ttl: int = 3600):
        self.redis = redis.from_url(redis_url)
        self.ttl = ttl

    def _key(self, prompt: str, model: str) -> str:
        content = f"{model}:{prompt}"
        return f"llm:cache:{hashlib.sha256(content.encode()).hexdigest()}"

    async def get(self, prompt: str, model: str) -> str | None:
        return await self.redis.get(self._key(prompt, model))

    async def set(self, prompt: str, model: str, response: str, ttl: int | None = None):
        await self.redis.setex(
            self._key(prompt, model),
            ttl or self.ttl,
            response
        )

    async def delete(self, prompt: str, model: str):
        await self.redis.delete(self._key(prompt, model))

    async def flush_pattern(self, pattern: str = "llm:cache:*"):
        keys = []
        async for key in self.redis.scan_iter(match=pattern):
            keys.append(key)
        if keys:
            await self.redis.delete(*keys)

    async def close(self):
        await self.redis.close()


class RedisEmbeddingCache:
    def __init__(self, redis_url: str = "redis://localhost:6379", ttl: int = 86400):
        self.redis = redis.from_url(redis_url)
        self.ttl = ttl

    async def get_or_compute(self, text: str, compute_fn, ttl: int | None = None) -> list[float]:
        key = f"emb:{hashlib.sha256(text.encode()).hexdigest()}"
        cached = await self.redis.get(key)
        if cached:
            return json.loads(cached)
        embedding = await compute_fn(text)
        await self.redis.setex(key, ttl or self.ttl, json.dumps(embedding))
        return embedding

    async def close(self):
        await self.redis.close()


# Upstash (serverless Redis) usage:
# pip install upstash-redis
# from upstash_redis import Redis
# redis = Redis.from_env()
"""

print("Redis/Upstash integration patterns shown above.")
print("Uncomment and configure for production use.")

## 8. Cache Invalidation Strategies

Choose the right invalidation approach for your data freshness requirements.

In [ ]:
class VersionedCache:
    """Cache with version-based invalidation."""

    def __init__(self, cache: InMemoryCache):
        self.cache = cache
        self.versions: dict[str, int] = defaultdict(int)

    def bump_version(self, namespace: str):
        """Invalidate all entries in a namespace."""
        self.versions[namespace] += 1

    def get(self, namespace: str, content: str) -> Optional[str]:
        version = self.versions[namespace]
        key = f"v{version}:{content}"
        return self.cache.get(namespace, key)

    def set(self, namespace: str, content: str, value: str, ttl: int = 3600):
        version = self.versions[namespace]
        key = f"v{version}:{content}"
        self.cache.set(namespace, key, value, ttl)


# Demo: version-based invalidation
versioned = VersionedCache(InMemoryCache(default_ttl=300))

# Store some data
versioned.set("docs", "page:1", "Old content v1")
versioned.set("docs", "page:2", "Old content v2")

print("Before invalidation:")
print(f"  page:1: {versioned.get('docs', 'page:1')}")
print(f"  page:2: {versioned.get('docs', 'page:2')}")

# Bump version to invalidate
versioned.bump_version("docs")

print("\nAfter invalidation:")
print(f"  page:1: {versioned.get('docs', 'page:1')}")
print(f"  page:2: {versioned.get('docs', 'page:2')}")

# Store new data
versioned.set("docs", "page:1", "Updated content v2")

print("\nAfter update:")
print(f"  page:1: {versioned.get('docs', 'page:1')}")

## Key Takeaways

1. **Response caching** eliminates redundant LLM calls — biggest single optimization
2. **Embedding caching** saves compute in RAG pipelines with repeated text
3. **Track hit rates** to tune TTLs and understand query patterns
4. **Use Redis/Upstash** for distributed caching across multiple instances
5. **Version-based invalidation** is simple and effective for most use cases